# News Trend Analysis — Interactive Notebook

This notebook walks you through the full pipeline interactively.
You can run each cell in order or tweak parameters and re-run.

**Pipeline:**
1. Fetch headlines from NewsAPI
2. Clean & analyse the text
3. Visualise trending keywords

> Make sure you have installed the requirements first:
> ```
> pip install -r requirements.txt
> ```

## 0. Setup — Imports & Configuration

In [ ]:
import os, re, sys
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

# Make sure imports from src/ work when running from the root folder
sys.path.insert(0, os.path.abspath('src'))

# ── Set your NewsAPI key here ──────────────────────────────────────────────
API_KEY = os.getenv('NEWS_API_KEY', '054a11f3fd634d5a90f9c09c25fd2ae6')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')

## 1. Fetch Headlines from NewsAPI

In [ ]:
def fetch_headlines(api_key, country='us', category='general', page_size=100):
    """Fetch top headlines and return a clean DataFrame."""
    url = 'https://newsapi.org/v2/top-headlines'
    params = {
        'apiKey': api_key,
        'country': country,
        'category': category,
        'pageSize': page_size,
    }
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    if data.get('status') != 'ok':
        raise ValueError(f"NewsAPI error: {data.get('message')}")

    articles = data.get('articles', [])
    rows = [{
        'title':        a.get('title', ''),
        'description':  a.get('description', ''),
        'source':       a.get('source', {}).get('name', ''),
        'published_at': a.get('publishedAt', ''),
        'url':          a.get('url', ''),
    } for a in articles]

    df = pd.DataFrame(rows)
    df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
    df.dropna(subset=['title'], inplace=True)
    return df.reset_index(drop=True)


df = fetch_headlines(API_KEY, country='us', category='general')
os.makedirs('data', exist_ok=True)
df.to_csv('data/headlines.csv', index=False)
print(f'Fetched {len(df)} headlines. Saved to data/headlines.csv')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Load from CSV (so this cell works even if you already ran fetch_news.py separately)
df = pd.read_csv('data/headlines.csv', parse_dates=['published_at'])
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.info()

In [ ]:
# Articles per source
source_counts = df['source'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 4))
source_counts.plot(kind='bar', ax=ax, color='#42A5F5', edgecolor='white')
ax.set_title('Top 10 News Sources by Article Count', fontweight='bold')
ax.set_xlabel('Source')
ax.set_ylabel('Articles')
ax.tick_params(axis='x', rotation=45)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Keyword Frequency Analysis

In [ ]:
STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','it','its','as','by','be','are','was','were','has','have','had',
    'that','this','from','not','he','she','we','they','you','i','his','her',
    'their','our','will','can','may','do','did','does','after','about','over',
    'into','up','out','more','new','says','said','say','us','than','so','no',
    'if','what','how','who','which','when','been','also','could','would',
    'should','then','now','all','one','two','just','get','still','amid',
}

def extract_words(df, stopwords):
    all_words = []
    for title in df['title'].dropna():
        text = re.sub(r'[^a-z\s]', ' ', title.lower())
        words = [w for w in text.split() if w not in stopwords and len(w) >= 3]
        all_words.extend(words)
    return all_words

words = extract_words(df, STOPWORDS)
counter = Counter(words)
freq_df = pd.DataFrame(counter.most_common(30), columns=['word', 'count'])
freq_df['percentage'] = (freq_df['count'] / sum(counter.values()) * 100).round(2)

# Save for visualise.py
freq_df.to_csv('data/word_freq.csv', index=False)

print(f'Total words: {len(words)} | Unique: {len(counter)}')
freq_df.head(15)

## 4. Visualise — Trending Keywords Bar Chart

In [ ]:
top_df = freq_df.head(20).iloc[::-1].reset_index(drop=True)

colors = ['#F44336' if i >= len(top_df) - 5 else '#2196F3' for i in range(len(top_df))]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top_df['word'], top_df['count'], color=colors, edgecolor='white')

for bar, count in zip(bars, top_df['count']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            str(count), va='center', ha='left', fontsize=9)

ax.set_xlabel('Frequency')
ax.set_title('Top 20 Trending Keywords in News Headlines', fontweight='bold', pad=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('data/trending_keywords.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to data/trending_keywords.png')

## 5. Word Cloud (optional — requires `wordcloud` package)

In [ ]:
try:
    from wordcloud import WordCloud

    wc = WordCloud(width=1200, height=600, background_color='white',
                   colormap='Blues', max_words=100)
    wc.generate_from_frequencies(dict(zip(freq_df['word'], freq_df['count'])))

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('News Keyword Word Cloud', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('data/wordcloud.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Word cloud saved to data/wordcloud.png')

except ImportError:
    print('wordcloud not installed. Run: pip install wordcloud')

---
**End of notebook.** Check the `data/` folder for the saved CSV files and chart images.